In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# coding a kernel SVM.
data=pd.read_csv('dataset/unlinear_data.csv')
label=pd.read_csv('dataset/unlinear_label_np.csv')
## trans to numpy array
X=data.values
y=label.values
print(X.shape,y.shape)
print(y)

(500, 2) (500, 1)
[[ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [

In [17]:
## kernel slection
## 1. guassian
## 2. polynomial
## use gaussian as example
def gaussian_kernel(X1, X2, sigma):
    
    X1_sq = np.sum(X1**2, axis=1).reshape(-1, 1)
    X2_sq = np.sum(X2**2, axis=1).reshape(1, -1)
    
    dists_sq = X1_sq + X2_sq - 2 * X1 @ X2.T
    return np.exp(-dists_sq / (2 * sigma**2))

## y=Sigma alpha_i y_i K(x_i,x)+b
## dual problem: maximize W(alpha)=Sigma alpha_i - 0.5 Sigma alpha_i alpha_j y_i y_j K(x_i,x_j)
## s.t. 0<=alpha_i<=C, Sigma alpha_i y_i=0
## use smo algorithm to solve the dual problem
def smo(X, y, C, sigma, max_iter):
    m, n = X.shape
    alpha = np.zeros(m)
    b = 0
    tol = 1e-3  # 容错度，不要用 C
    
    # 【关键优化】提前计算全量核矩阵
    K_matrix = gaussian_kernel(X, X, sigma)
    
    for it in range(max_iter):
        alpha_pairs_changed = 0
        for i in range(m):
            # 计算样本 i 的误差 E_i
            # f_i = Sigma alpha_j * y_j * K_ij + b
            f_i = np.sum(alpha * y * K_matrix[:, i]) + b
            E_i = f_i - y[i]
            
            # 【核心修正】修正 KKT 判断逻辑
            if ((y[i] * E_i < -tol and alpha[i] < C) or (y[i] * E_i > tol and alpha[i] > 0)):
                # 随机选择第二个变量 j
                j = np.random.choice([x for x in range(m) if x != i])
                
                f_j = np.sum(alpha * y * K_matrix[:, j]) + b
                E_j = f_j - y[j]
                
                alpha_i_old, alpha_j_old = alpha[i].copy(), alpha[j].copy()
                
                # 计算 L 和 H 边界
                if y[i] != y[j]:
                    L = max(0, alpha[j] - alpha[i])
                    H = min(C, C + alpha[j] - alpha[i])
                else:
                    L = max(0, alpha[i] + alpha[j] - C)
                    H = min(C, alpha[i] + alpha[j])
                
                if L == H: continue
                
                # 计算 eta (学习率/曲率)
                eta = 2.0 * K_matrix[i, j] - K_matrix[i, i] - K_matrix[j, j]
                if eta >= 0: continue
                
                # 更新 alpha[j]
                alpha[j] -= y[j] * (E_i - E_j) / eta
                alpha[j] = np.clip(alpha[j], L, H)
                
                # 如果变化太小，跳过
                if abs(alpha[j] - alpha_j_old) < 1e-5: continue
                
                # 更新 alpha[i]
                alpha[i] += y[i] * y[j] * (alpha_j_old - alpha[j])
                
                # 更新偏置 b
                b1 = b - E_i - y[i]*(alpha[i]-alpha_i_old)*K_matrix[i,i] - y[j]*(alpha[j]-alpha_j_old)*K_matrix[i,j]
                b2 = b - E_j - y[i]*(alpha[i]-alpha_i_old)*K_matrix[i,j] - y[j]*(alpha[j]-alpha_j_old)*K_matrix[j,j]
                
                if 0 < alpha[i] < C: b = b1
                elif 0 < alpha[j] < C: b = b2
                else: b = (b1 + b2) / 2
                
                alpha_pairs_changed += 1
        
        # 监控进度
        if it % 10 == 0:
            print(f"Iteration {it}, changed pairs: {alpha_pairs_changed}")
            
        if alpha_pairs_changed == 0:
            print(f"Converged at iteration {it}")
            break
            
    return alpha, b

def main():
    ## X.shape=(500,2),y.shape=(500,1)
    C=1.0
    sigma=1.0
    max_iter=100
    alpha,b=smo(X,y,C,sigma,max_iter)
    K=gaussian_kernel(X,X,sigma)
    y_flatten=y.reshape(-1)
    y_pred=np.sign(K@(alpha*y_flatten)+b)
    print("Accuracy:",np.mean(y_pred==y.flatten()))
    print(y_pred)


if __name__=="__main__":
    main()


/var/folders/q1/nxf4tks52ms9xvctj_k1b9fr0000gn/T/ipykernel_24811/3181147265.py:59: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  alpha[j] -= y[j] * (E_i - E_j) / eta
/var/folders/q1/nxf4tks52ms9xvctj_k1b9fr0000gn/T/ipykernel_24811/3181147265.py:66: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  alpha[i] += y[i] * y[j] * (alpha_j_old - alpha[j])


Iteration 0, changed pairs: 131
Iteration 10, changed pairs: 2
Converged at iteration 18
Accuracy: 0.974
[ 1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1. -1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1.  1.  1.  1.  1.  1.  1. -1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1.  1.  1.  1.  1. -1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1.  1.  1.  1.  1.  1.  1.  1. -1.  1.  1.  1.  1.  1. -1.  1.  1.  1.
  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1. -1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.
  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1. -1.
  1.  1.  1.  1.  1